# ORSO Example: 2-Layer NF Inference

Smoke-test the 2-layer normalizing-flow checkpoint on `exp_data/ORSO_example.ort`.

The workflow loads the measured reflectivity curve, samples the posterior under explicit two-layer priors, then inspects the MAP curve and sampled SLD profiles. ORSO supplies no numeric ground-truth SLD profile, so these plots are diagnostic rather than an accuracy comparison.

## 1. Setup

Find the repository root so the local model, data, and project helpers work when the notebook is opened from `notebooks/`. The plotting backend is set explicitly for inline notebook output.

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib as mpl

mpl.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
import torch

# Locate the repository root from either the project root or notebooks/.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "exp_data" / "ORSO_example.ort").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the inverse-eval repo.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from device_utils import detect_torch_device, summarize_torch_backends
from nf_statistics import compute_nf_sample_statistics
from reflectorch import EasyInferenceModel


# Convert model outputs to NumPy before indexing, statistics, and plotting.
def to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def extend_prior_bounds_for_nf(prior_bounds):
    prior_bounds = list(prior_bounds)
    if len(prior_bounds) in (5, 8):
        return [*prior_bounds, (0.9, 1.1), (-10.0, -4.0)]
    if len(prior_bounds) in (7, 10):
        return prior_bounds
    raise ValueError(f"Unsupported NF prior_bounds length={len(prior_bounds)}")

plt.rcParams["text.usetex"] = False
torch.manual_seed(42)

print(f"Project root    : {PROJECT_ROOT}")
print(f"PyTorch version : {torch.__version__}")
print(f"Backends        : {summarize_torch_backends()}")

## 2. Configure Model and Priors

The L2 checkpoint expects eight structural parameters plus reflectivity scale and background. The bounds below encode the Si/SiO2/polymer/D2O stack and are passed directly to the NF sampler.

In [ ]:
ORSO_FILE = PROJECT_ROOT / "exp_data" / "ORSO_example.ort"
VENDOR_ROOT = PROJECT_ROOT / "vendor" / "nflows_reflectorch"

NF_CONFIG = "example_nf_config_reflectorch_L2.yaml"
NF_SAMPLES = 1000  # increase for final plots
DEVICE = detect_torch_device()
# The ORSO/Anaklasis model is measured from the Si (fronting) side.
AMBIENT_SLD = 2.07

# Absolute SLD bounds. sample() subtracts AMBIENT_SLD before inference.
# Layer order: Si | SiO2 (L2) | polymer (L1) | D2O (sub).
STRUCTURAL_PRIOR_BOUNDS = [
    (1.0, 100.0),   # Thickness L2 / SiO2 [A]
    (1.0, 700.0),   # Thickness L1 / polymer [A]
    (0.0, 5.0),     # Roughness Si/SiO2 [A]
    (0.0, 50.0),    # Roughness SiO2/polymer [A]
    (0.0, 60.0),    # Roughness polymer/D2O [A]
    (3.46, 3.48),   # SLD SiO2 [1e-6 A^-2]
    (-0.5, 3.0),    # SLD polymer [1e-6 A^-2]
    (6.34, 6.36),   # SLD D2O [1e-6 A^-2]
]

PRIOR_BOUNDS = [
    *STRUCTURAL_PRIOR_BOUNDS,
    (0.9, 1.1),     # Reflectivity scale
    (-10.0, -6.0),  # log10 background [reflectivity]
]

print(f"ORSO file      : {ORSO_FILE}")
print(f"NF config      : {NF_CONFIG}")
print(f"Device         : {DEVICE}")
print(f"Samples        : {NF_SAMPLES}")
print(f"Prior dim      : {len(PRIOR_BOUNDS)}")

## 3. Load ORSO Data

Read Q, reflectivity, uncertainty, and optional Q-resolution columns from the ORSO export. Non-finite or non-positive measurements are removed before inference.

In [ ]:
def load_orso_curve(path: Path):
    data = np.loadtxt(path, comments="#")
    if data.ndim != 2 or data.shape[1] < 3:
        raise ValueError(f"Expected at least Q, R, dR columns in {path}")

    q = data[:, 0]
    r = data[:, 1]
    dr = data[:, 2]
    dq = data[:, 3] if data.shape[1] >= 4 else None

    # Keep only measurements valid for log-scale reflectivity inference.
    mask = np.isfinite(q) & np.isfinite(r) & np.isfinite(dr) & (q > 0) & (r > 0) & (dr > 0)
    if dq is not None:
        mask &= np.isfinite(dq) & (dq >= 0)
        dq = dq[mask]

    return q[mask], r[mask], dr[mask], dq


q_exp, r_exp, dr_exp, dq_exp = load_orso_curve(ORSO_FILE)

print(f"Loaded points : {len(q_exp)}")
print(f"Q range       : {q_exp.min():.5f} - {q_exp.max():.5f} A^-1")
print(f"R range       : {r_exp.min():.3e} - {r_exp.max():.3e}")
print(f"dR/R range    : {(dr_exp / r_exp).min():.3f} - {(dr_exp / r_exp).max():.3f}")
if dq_exp is not None:
    print(f"dQ/Q median   : {np.median(dq_exp / q_exp):.4f}")

assert len(q_exp) > 0
assert np.all(np.diff(q_exp) > 0)

## 4. Run Inference

`EasyInferenceModel` loads the vendored L2 checkpoint and interpolates the measured curve onto its model grid. Sampling returns posterior parameters, reflectivity curves, SLD profiles, and log-likelihoods.

In [ ]:
model = EasyInferenceModel(
    config_name=NF_CONFIG,
    root_dir=str(VENDOR_ROOT),
    repo_id=None,
    device=DEVICE,
)

expected_dim = model.trainer.loader.prior_sampler.param_dim
assert len(PRIOR_BOUNDS) == expected_dim, (len(PRIOR_BOUNDS), expected_dim)

# Pass the measured dQ values when available; otherwise use the model default.
prediction = model.preprocess_and_sample(
    reflectivity_curve=r_exp,
    q_values=q_exp,
    sigmas=dr_exp,
    q_resolution=dq_exp if dq_exp is not None else 0.1,
    ambient_sld=AMBIENT_SLD,
    prior_bounds=PRIOR_BOUNDS,
    num_samples=NF_SAMPLES,
    sampling_batch_size=128,
    maximum_sim_batch_size=128,
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
    calc_log_likelihoods=True,
    enable_importance_sampling=True,
    clip_prediction=True,
)

print("Inference complete.")
print(f"Sampled params : {prediction['predicted_params_array'].shape}")
print(f"Sampled curves : {prediction['sampled_curves'].shape}")
print(f"Param names    : {prediction['param_names']}")

## 5. Inspect the MAP Sample

The sample with the largest log-likelihood is used as the MAP estimate. Summary statistics over all samples remain useful for showing posterior spread.

In [ ]:
log_likelihoods = to_numpy(prediction["log_likelihoods"])
samples = to_numpy(prediction["predicted_params_array"]).copy()
param_names = prediction["param_names"]
# sample() returns SLD parameters relative to the fronting SLD.
if AMBIENT_SLD is not None:
    samples[:, [i for i, name in enumerate(param_names) if name.startswith("SLD")]] += AMBIENT_SLD
best_idx = int(np.argmax(log_likelihoods))
map_params = samples[best_idx]
stats = compute_nf_sample_statistics(samples, log_likelihoods)

print(f"MAP index       : {best_idx}")
print(f"MAP loglike     : {log_likelihoods[best_idx]:.4f}")
if "ess" in prediction:
    print(f"ESS             : {prediction['ess']:.1f} / {NF_SAMPLES}")
print()
print(f"{'Parameter':<22} {'Prior min':>10} {'MAP':>12} {'Mean':>12} {'Std':>12} {'Prior max':>10}")
print("-" * 84)
for i, name in enumerate(param_names):
    lo, hi = PRIOR_BOUNDS[i]
    print(
        f"{name:<22} {lo:>10.3f} {map_params[i]:>12.4f} "
        f"{stats['nf_params_mean'][i]:>12.4f} {stats['nf_params_std'][i]:>12.4f} {hi:>10.3f}"
    )

assert samples.shape == (NF_SAMPLES, expected_dim)
assert np.isfinite(map_params).all()

## 6. Plot the MAP Fit and SLD

The left panel compares the measured curve with the MAP prediction; the right panel shows its corresponding SLD profile. Neither panel is a comparison to a supplied ground-truth profile.

In [ ]:
q_plot = to_numpy(prediction["q_plot_pred"])
curve_map = to_numpy(prediction["sampled_curves"])[best_idx]
sld_x = to_numpy(prediction["sampled_sld_xaxis"])
sld_map = to_numpy(prediction["sampled_sld_profiles"])[best_idx]

fig, (ax_r, ax_sld) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

ax_r.errorbar(q_exp, r_exp, yerr=dr_exp, fmt=".", ms=3, alpha=0.45, label="ORSO")
ax_r.plot(q_plot, curve_map, lw=2, label="2-layer NF MAP")
ax_r.set(xlabel="Q (A^-1)", ylabel="Reflectivity", yscale="log")
ax_r.legend()

ax_sld.plot(sld_x, sld_map, lw=2)
ax_sld.set(xlabel="Depth (A)", ylabel="SLD (1e-6 A^-2)")

plt.show()

## 7. NF Sampled SLD Profiles

Overlay every finite SLD profile sampled by the NF posterior. Multiple separated profile families for the same reflectivity curve indicate an ill-posed inverse result.

In [ ]:
def finite_sld_profiles(prediction):
    sld_x = to_numpy(prediction["sampled_sld_xaxis"])
    profiles = to_numpy(prediction["sampled_sld_profiles"])
    loglike = to_numpy(prediction["log_likelihoods"]).ravel()
    if profiles.ndim == 1:
        profiles = profiles[None, :]
    # Invalid posterior draws cannot be plotted or used to choose the MAP profile.
    mask = np.isfinite(loglike) & np.all(np.isfinite(profiles), axis=1)
    profiles = profiles[mask]
    loglike = loglike[mask]
    if sld_x.ndim == 2:
        sld_x = sld_x[mask]
    if len(profiles) == 0:
        raise ValueError("No finite sampled SLD profiles available to plot.")
    return sld_x, profiles, loglike


sld_x_all, sld_profiles_all, sld_loglike_all = finite_sld_profiles(prediction)
best_sld_idx = int(np.argmax(sld_loglike_all))
order = np.argsort(sld_loglike_all)
line_alpha = min(0.18, max(0.01, 30 / len(order)))

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
for idx in order:
    x = sld_x_all[idx] if np.ndim(sld_x_all) == 2 else sld_x_all
    ax.plot(x, sld_profiles_all[idx], color="0.25", alpha=line_alpha, lw=0.8)

x_best = sld_x_all[best_sld_idx] if np.ndim(sld_x_all) == 2 else sld_x_all
ax.plot(x_best, sld_profiles_all[best_sld_idx], color="crimson", lw=2.4, label="MAP")
ax.set(xlabel="Depth (A)", ylabel="SLD (1e-6 A^-2)", title="ORSO NF sampled SLD profiles")
ax.legend()
plt.show()

print(f"Plotted {len(sld_profiles_all)} finite sampled SLD profiles.")


## Try it yourself: check the effect of sample count

Compare `NF_SAMPLES = 500` and `NF_SAMPLES = 2000` for the same ORSO curve and prior bounds. For each setting, rerun the notebook from Setup through the sampled-SLD-profiles plot so the fixed random seed is reset. Compare the MAP parameters, posterior means and standard deviations, and the spread of the overlaid SLD profiles.

**Explain:** Which results look stable as the number of samples increases? Do additional samples change the visible profile families, or mainly fill them in more densely? This is a sampling-stability check for this run, not a guarantee that the posterior or the model is correct.
